#Mounting the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Installing All Dependencies

In [ ]:
!pip install pyyaml==5.1
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())
!gcc --version
# opencv is pre-installed on colab

# install detectron2: (Colab has CUDA 10.1 + torch 1.7)
# See https://detectron2.readthedocs.io/tutorials/install.html for instructions
import torch
assert torch.__version__.startswith("1.7")
!pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu101/torch1.7/index.html

# Restart the runtime after running the above cell
# For restarting runtime go to Runtime->Restart Runtime 

#Importing All the Libraries

In [ ]:
from detectron2.data.datasets import register_coco_instances
import os
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
import numpy as np
import cv2
import matplotlib.pyplot as plt
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.utils.visualizer import ColorMode
import random
import json
import pickle
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import ColorMode
from detectron2.data import DatasetCatalog, MetadataCatalog
from google.colab.patches import cv2_imshow

#Load all the configurations and weights

In [ ]:
%cd /content/drive/MyDrive/Detectron/

/content/drive/MyDrive/Detectron


In [ ]:
%pwd

'/content/drive/My Drive/Detectron'

In [ ]:
from detectron2.data.datasets import register_coco_instances

for d in ["train", "test"]:
    register_coco_instances(f"f1_{d}", {}, f"/content/drive/MyDrive/FinalDental/{d}.json", f"/content/drive/MyDrive/FinalDental/{d}")

In [ ]:
import os
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
#cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/panoptic_fpn_R_101_3x.yaml"))
cfg.DATASETS.TRAIN = ("f1_train",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
#cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/panoptic_fpn_R_101_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.001
cfg.SOLVER.MAX_ITER = 15000
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 32

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
#import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
trainer = DefaultTrainer(cfg) 
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
from detectron2.config import get_cfg
cfg = get_cfg()

cfg.merge_from_file("/content/config.yaml") 
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.8
cfg.merge_from_list(["MODEL.WEIGHTS", "/content/drive/MyDrive/Detectron/output/model_final.pth"])
predictor = DefaultPredictor(cfg)  
print(cfg.dump())

In [ ]:
import pickle
a_file = open(os.path.join("/content/meta.pkl"), "rb")
MetadataCatalog = pickle.load(a_file)

In [ ]:
MetadataCatalog.get(cfg.DATASETS.TRAIN[0]).get('thing_classes')

In [ ]:
image = cv2.imread("/content/01711030003-opg.jpg")
outputs = predictor(image)
print(outputs)
v = Visualizer(image[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)

v = v.draw_instance_predictions(outputs["instances"].to("cpu"))
plt.figure(figsize = (14, 10))
plt.imshow(cv2.cvtColor(v.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
outputs["instances"].pred_masks.shape

In [ ]:
classes=outputs['instances'].pred_classes.cpu().numpy()
print(type(classes))
print(classes)

In [ ]:
boxes=outputs['instances'].pred_boxes.tensor.cpu().numpy()
print(type(boxes))
print(boxes)

In [ ]:
areas = np.prod(boxes[:, 2:] - boxes[:, :2], axis=1)
sorted_idxs = np.argsort(-areas).tolist()

In [ ]:
masks=outputs['instances'].pred_masks.cpu().numpy()
print(masks)

In [ ]:
masks = masks[sorted_idxs]
masks = 1*masks
  
print(masks)

In [ ]:
class_names = MetadataCatalog.get(cfg.DATASETS.TRAIN[0]).get('thing_classes')
classes = outputs["instances"].pred_classes.cpu().numpy().tolist()
print(classes)

[18, 26, 20, 16, 14, 6, 22, 3, 9, 12, 15, 24, 10, 25, 13, 5, 27, 23, 17, 7, 11, 31, 4, 21, 0, 2, 30]


In [ ]:
labels = [class_names[i] for i in classes]
print(labels)

['30', '19', '31', '18', '29', '22', '16', '32', '21', '28', '3', '20', '25', '15', '26', '14', '4', '10', '1', '27', '5', '8', '17', '2', '9', '23', '6']


In [ ]:
boxes = boxes[sorted_idxs]
labels = [labels[k] for k in sorted_idxs]

In [ ]:
dict_ = dict(zip(labels,boxes))

In [ ]:
print(dict_)

{'30': array([375.02765, 328.94745, 448.84158, 428.21066], dtype=float32), '32': array([248.00333, 308.25012, 332.91977, 393.77417], dtype=float32), '31': array([313.50555, 322.7494 , 388.83047, 413.79782], dtype=float32), '19': array([715.4839 , 335.8304 , 781.77075, 437.32214], dtype=float32), '18': array([768.12396, 333.5717 , 832.9069 , 428.49368], dtype=float32), '3': array([341.58527, 226.19595, 401.13858, 323.8703 ], dtype=float32), '17': array([814.6322 , 324.03348, 882.76556, 408.65668], dtype=float32), '14': array([730.8868 , 236.5817 , 780.595  , 336.36157], dtype=float32), '1': array([251.87749, 209.0398 , 304.54388, 299.9694 ], dtype=float32), '2': array([299.9052 , 216.71193, 347.90536, 313.9167 ], dtype=float32), '29': array([435.60864, 334.2677 , 480.79364, 434.31168], dtype=float32), '15': array([776.13873, 231.97618, 820.74036, 331.20544], dtype=float32), '9': array([519.3441 , 234.538  , 565.893  , 325.83426], dtype=float32), '22': array([633.1223 , 346.2941 , 672.49

In [ ]:
from PIL import Image
import cv2
import os
for i,j in zip(range(len(dict_)),dict_):
  image = cv2.imread("/content/01711030003-opg.jpg")
  image = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
  image = image*masks[i]
  cv2.imwrite('mask'+str(j)+'.jpg', image)
  img = Image.open('mask'+str(j)+'.jpg')
  img = img.convert('RGBA')
  x1 = dict_[j][0]
  y1 = dict_[j][1]
  x2 = dict_[j][2]
  y2 = dict_[j][3]
  area = (x1,y1,x2,y2)
  cropped_img = img.crop(area)
  cropped_img.save(str(j)+'.png')


In [ ]:
import os
for files in os.listdir('/content/'):
  if files.startswith('mask'):
    os.remove(files)  